In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE


df = pd.read_csv("D:/Đại học/1. Nghiên cứu Khoa học/Dataset/Folder_Dataset/P2P_Dataset.csv")
boolean_cols = df.select_dtypes(include=['bool']).columns
df[boolean_cols] = df[boolean_cols].astype(int)
df = df.drop(columns=["badloan", "funded_amnt", "sub_grade", "pub_rec", "num_tl_30dpd"])


In [ ]:
# 🎯 Chia X, y
X = df.drop(columns=["default_binary"], errors="ignore")
y = df["default_binary"]

# 🔀 Chia tập train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50, stratify=y)

# ⚖️ Cân bằng dữ liệu bằng SMOTE
smote = SMOTE(sampling_strategy=0.5, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)


from catboost import CatBoostClassifier
# 🚀 Huấn luyện mô hình CatBoost cơ bản
cat_model = CatBoostClassifier(iterations=100, random_state=42, verbose=0)
cat_model.fit(X_train_resampled, y_train_resampled)

# 🔍 Lấy độ quan trọng của từng biến
importance_cat = cat_model.get_feature_importance()

# 📊 Hiển thị tầm quan trọng của biến
for i, v in enumerate(importance_cat):
    print(f'Biến: {X_train.columns[i]}, Tầm quan trọng: {v:.5f}')

# 🎨 Vẽ biểu đồ Feature Importance
plt.figure(figsize=(12, 6))
plt.bar([x for x in range(len(importance_cat))], importance_cat)
plt.xticks(range(len(importance_cat)), X_train.columns, rotation=90)
plt.xlabel("Features")
plt.ylabel("Importance Score")
plt.title("Feature Importance từ CatBoost")
plt.show()


# 🎯 Chỉ chọn các biến có tầm quan trọng > 0.01
important_features_cat = X_train.columns[importance_cat > 0.01]

# 🔄 Cập nhật X_train và X_test chỉ với các biến quan trọng
X_train_selected = X_train_resampled[important_features_cat]
X_test_selected = X_test[important_features_cat]

# 🔄 Chuẩn hóa dữ liệu
scaler = StandardScaler()
X_train_selected_scaled = scaler.fit_transform(X_train_selected)
X_test_selected_scaled = scaler.transform(X_test_selected)

In [ ]:
selected_features_catboost = [
    "acc_open_past_24mths", "annual_inc", "delinq_2yrs", "dti", "earnings",
    "emp_length", "grade", "inq_last_12m", "int_rate", "loan_amnt",
    "loan_vol6m", "mths_since_last_delinq", "pct_tl_nvr_dlq", "revol_util",
    "term", "home_ownership_MORTGAGE", "home_ownership_OWN", "home_ownership_RENT",
    "purpose_credit_card", "purpose_debt_consolidation", "purpose_home_improvement",
    "purpose_other"
]

X_train_selected_catboost = X_train_resampled[selected_features_catboost]
X_test_selected_catboost = X_test[selected_features_catboost]

scaler = StandardScaler()
X_train_selected_catboost_scaled = scaler.fit_transform(X_train_selected_catboost)
X_test_selected_catboost_scaled = scaler.transform(X_test_selected_catboost)


In [ ]:
# 🛠 **Định nghĩa tập siêu tham số cần tìm**
param_dist = {
    "iterations": [100, 300, 500, 700, 1000],
    "depth": [3, 5, 7, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3],
    "l2_leaf_reg": [1, 3, 5, 7, 10],
    "border_count": [32, 64, 128, 255],
    "random_strength": [1, 3, 5, 7, 10],
}

# 🚀 **Khởi tạo mô hình CatBoost**
cat_model = CatBoostClassifier(random_state=42, verbose=0)

# 🎯 **Tìm siêu tham số tốt nhất với RandomizedSearchCV**
random_search = RandomizedSearchCV(
    estimator=cat_model,
    param_distributions=param_dist,
    n_iter=15,  # Giảm số lần thử nghiệm ngẫu nhiên để tăng tốc độ
    scoring="roc_auc",
    cv=3,  # Giảm số lần cross-validation để nhanh hơn
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_selected_catboost_scaled, y_train_resampled)

# 🔥 **Lấy siêu tham số tối ưu**
best_params = random_search.best_params_
print("\n🔥 Best Hyperparameters for CatBoost:")
print(best_params)


🔥 Best Hyperparameters for CatBoost:
{'random_strength': 5, 'learning_rate': 0.3, 'l2_leaf_reg': 5, 'iterations': 300, 'depth': 5, 'border_count': 255}


In [ ]:
# 🚀 **Huấn luyện mô hình với tham số tối ưu**
best_cat = CatBoostClassifier(**best_params, random_state=42, verbose=0)
best_cat.fit(X_train_selected_catboost_scaled, y_train_resampled)

# 📊 **Dự đoán trên tập kiểm tra**
y_pred = best_cat.predict(X_test_selected_catboost_scaled)
y_prob = best_cat.predict_proba(X_test_selected_catboost_scaled)[:, 1]

# 🎯 **Tính các chỉ số đánh giá**
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
conf_matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred)

# 📊 **In kết quả**
print("\n📊 CatBoost Model Evaluation")
print(f"🔥 Best Hyperparameters: {best_params}")
print(f"🎯 Accuracy: {accuracy:.4f}")
print(f"🚀 AUC-ROC: {roc_auc:.4f}")
print(f"📑 Classification Report:\n{report}")
print(f"📊 Confusion Matrix:\n{conf_matrix}")


📊 CatBoost Model Evaluation
🔥 Best Hyperparameters: {'random_strength': 5, 'learning_rate': 0.3, 'l2_leaf_reg': 5, 'iterations': 300, 'depth': 5, 'border_count': 255}
🎯 Accuracy: 0.9137
🚀 AUC-ROC: 0.8148
📑 Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.99      0.95    494039
           1       0.50      0.07      0.13     46647

    accuracy                           0.91    540686
   macro avg       0.71      0.53      0.54    540686
weighted avg       0.88      0.91      0.88    540686

📊 Confusion Matrix:
[[490592   3447]
 [ 43189   3458]]
